# UMUD v2 – Structure-Tensor Pipeline

**Strategy:** Segment aponeuroses only (ResNet34 U-Net) → fit two lines in original pixels → MT from line distance → PA from structure tensor of the inter-aponeurosis band → FL = MT / sin(PA).

Muscle-triangle: PA = atan(MT/dx), FL = sqrt(MT^2 + dx^2) = MT/sin(PA)

In [ ]:

import os, math, warnings, cv2
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models as tvm
from scipy import ndimage
from PIL import Image
warnings.filterwarnings("ignore")

def seed_everything(s=42):
    np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s); torch.backends.cudnn.deterministic = True

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}  Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def walk_find(root, name, is_dir=True):
    for cur, dirs, files in os.walk(root):
        if name in (dirs if is_dir else files):
            return os.path.join(cur, name)
    return None

SEARCH_ROOTS = ["/kaggle/input", "/content", "."]

def find(name, is_dir=True):
    for r in SEARCH_ROOTS:
        if os.path.exists(r):
            p = walk_find(r, name, is_dir)
            if p: return p
    return None

APO_IMG_DIR  = find("apo_images_new_model_v1")
APO_MSK_DIR  = find("apo_masks_new_model_v1")
TEST_IMG_DIR = find("test_set_v2")

SCALE_CSV = None
for r in SEARCH_ROOTS:
    if not os.path.exists(r): continue
    for cur, dirs, files in os.walk(r):
        for f in files:
            if "universal_scale_metadata" in f.lower() and f.endswith(".csv"):
                SCALE_CSV = os.path.join(cur, f)
                break
        if SCALE_CSV: break

for k, v in [("APO_IMG", APO_IMG_DIR), ("APO_MSK", APO_MSK_DIR),
             ("TEST_IMG", TEST_IMG_DIR), ("SCALE_CSV", SCALE_CSV)]:
    print(f"  [{k}] {v}")


In [ ]:

# Columns used from universal_scale_metadata_v2.csv:
#   record_id, apo_mm_per_pixel, fasc_mm_per_pixel,
#   apo_spacing (pixels per 10 mm tick),
#   apo_depth_mm, fasc_depth_mm

def res_fallback_mmpx(w, h):
    if w == 1200 and h == 800:  return 50.0 / 720.0
    if w == 1640 and h == 1080: return 45.0 / 980.0
    if w in (1088, 1069, 959):  return 45.0 / 600.0
    if 450 <= w <= 470:         return 45.0 / 460.0
    return 50.0 / max(h * 0.90, 1)

SCALE_MAP = {}
N_FROM_CSV = 0

if SCALE_CSV and os.path.exists(SCALE_CSV):
    sdf = pd.read_csv(SCALE_CSV)
    print(f"Scale CSV: {len(sdf)} rows  cols: {list(sdf.columns)}")
    for _, row in sdf.iterrows():
        stem = str(row["record_id"]).strip()
        mmpx = None
        for col in ("fasc_mm_per_pixel", "apo_mm_per_pixel"):
            v = row.get(col, np.nan)
            if pd.notna(v) and float(v) > 0:
                mmpx = float(v); break
        if mmpx is None:
            for col in ("fasc_spacing", "apo_spacing"):
                sp = row.get(col, np.nan)
                if pd.notna(sp) and float(sp) > 0:
                    mmpx = 10.0 / float(sp); break
        if mmpx is None:
            for col in ("fasc_depth_mm", "apo_depth_mm"):
                d = row.get(col, np.nan)
                if pd.notna(d) and float(d) > 0:
                    mmpx = ("depth", float(d)); break
        if mmpx is not None:
            SCALE_MAP[stem] = mmpx
            if isinstance(mmpx, float): N_FROM_CSV += 1
    print(f"  Direct mm/px: {N_FROM_CSV}  Depth-only: {len(SCALE_MAP)-N_FROM_CSV}")
else:
    print("[WARN] Scale CSV not found -- using resolution fallbacks.")

def get_mmpx(stem, orig_w, orig_h):
    val = SCALE_MAP.get(stem)
    if val is None:
        return res_fallback_mmpx(orig_w, orig_h)
    if isinstance(val, tuple):
        return val[1] / max(orig_h * 0.90, 1)
    return float(val)


In [ ]:

def list_imgs(folder):
    if not folder or not os.path.exists(folder): return []
    return sorted(f for f in os.listdir(folder)
                  if not f.lower().startswith("thumbs"))

apo_imgs  = set(list_imgs(APO_IMG_DIR))
apo_msks  = set(list_imgs(APO_MSK_DIR))
test_imgs = set(list_imgs(TEST_IMG_DIR))

records = []
for fname in sorted(apo_imgs & apo_msks):
    with Image.open(os.path.join(APO_IMG_DIR, fname)) as im: w, h = im.size
    records.append(dict(image_id=fname, split="train", has_apo=True,
                        orig_w=w, orig_h=h))

for fname in sorted(test_imgs):
    with Image.open(os.path.join(TEST_IMG_DIR, fname)) as im: w, h = im.size
    records.append(dict(image_id=fname, split="test", has_apo=False,
                        orig_w=w, orig_h=h))

meta = pd.DataFrame(records)
print(meta["split"].value_counts().to_string())
print("\nTest resolution breakdown:")
print(meta[meta.split == "test"].groupby(["orig_w","orig_h"])["image_id"].count().to_string())

n_fb = sum(1 for _, r in meta[meta.split == "test"].iterrows()
           if not isinstance(SCALE_MAP.get(os.path.splitext(r.image_id)[0]), float))
print(f"\nTest images needing fallback scale: {n_fb}/{(meta.split=='test').sum()}")


In [ ]:

IMG_H, IMG_W = 384, 576   # 2:3 ratio -- preserves angles

def load_img_tensor(path):
    img = np.array(Image.open(path).convert("L"), dtype=np.float32) / 255.0
    img = cv2.resize(img, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR)
    return torch.tensor(img).unsqueeze(0)

def load_mask_np(path):
    m = np.array(Image.open(path).convert("L"), dtype=np.uint8)
    # Critical fix: normalize inverted masks where background is white (>50% positive)
    if (m > 128).mean() > 0.5:
        m = (m == 0).astype(np.uint8) * 255
    m = cv2.resize(m, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    return (m > 64).astype(np.float32)

def augment_pair(img_np, msk_np):
    if np.random.rand() < 0.5:
        img_np = img_np[:, ::-1].copy()
        msk_np = msk_np[:, ::-1].copy()
    angle = np.random.uniform(-8, 8)
    M = cv2.getRotationMatrix2D((IMG_W/2, IMG_H/2), angle, 1.0)
    img_np = cv2.warpAffine(img_np, M, (IMG_W, IMG_H),
                            flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    msk_np = cv2.warpAffine(msk_np, M, (IMG_W, IMG_H),
                            flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT)
    g = np.random.uniform(0.75, 1.35)
    img_np = np.clip(img_np ** g * np.random.uniform(0.85, 1.15)
                     + np.random.uniform(-0.08, 0.08), 0, 1).astype(np.float32)
    return img_np, msk_np.astype(np.float32)

class ApoDataset(Dataset):
    def __init__(self, df, aug=False):
        self.df = df.reset_index(drop=True)
        self.aug = aug

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname  = row.image_id
        is_test = row.split == "test"
        img_dir = TEST_IMG_DIR if is_test else APO_IMG_DIR

        img_np = np.array(Image.open(os.path.join(img_dir, fname)).convert("L"),
                          dtype=np.float32) / 255.0
        img_np = cv2.resize(img_np, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR)

        if is_test:
            return torch.tensor(img_np).unsqueeze(0), fname, int(row.orig_w), int(row.orig_h)

        msk_np = load_mask_np(os.path.join(APO_MSK_DIR, fname))
        if self.aug:
            img_np, msk_np = augment_pair(img_np, msk_np)
        return torch.tensor(img_np).unsqueeze(0), torch.tensor(msk_np).unsqueeze(0)


In [ ]:

class ConvBnRelu(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.m = nn.Sequential(
            nn.Conv2d(ci, co, 3, padding=1, bias=False),
            nn.BatchNorm2d(co), nn.ReLU(inplace=True))
    def forward(self, x): return self.m(x)

class DecoderBlock(nn.Module):
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_c, out_c, 2, stride=2)
        self.conv = nn.Sequential(ConvBnRelu(out_c+skip_c, out_c), ConvBnRelu(out_c, out_c))
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = nn.functional.interpolate(x, size=skip.shape[2:],
                                          mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class UNetR34(nn.Module):
    def __init__(self):
        super().__init__()
        enc = tvm.resnet34(weights=tvm.ResNet34_Weights.DEFAULT)
        w0  = enc.conv1.weight.data.mean(dim=1, keepdim=True)
        enc.conv1 = nn.Conv2d(1, 64, 7, stride=2, padding=3, bias=False)
        enc.conv1.weight.data = w0
        self.e0   = nn.Sequential(enc.conv1, enc.bn1, enc.relu)
        self.pool = enc.maxpool
        self.e1, self.e2 = enc.layer1, enc.layer2
        self.e3, self.e4 = enc.layer3, enc.layer4
        self.d4 = DecoderBlock(512, 256, 256)
        self.d3 = DecoderBlock(256, 128, 128)
        self.d2 = DecoderBlock(128,  64,  64)
        self.d1 = DecoderBlock( 64,  64,  32)
        self.up = nn.ConvTranspose2d(32, 32, 2, stride=2)
        self.head = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e0 = self.e0(x); ep = self.pool(e0)
        e1 = self.e1(ep); e2 = self.e2(e1)
        e3 = self.e3(e2); e4 = self.e4(e3)
        d = self.d4(e4, e3); d = self.d3(d, e2)
        d = self.d2(d, e1);  d = self.d1(d, e0)
        return self.head(self.up(d))

class BCEDiceLoss(nn.Module):
    def __init__(self, pos_w=2.0, smooth=1e-5):
        super().__init__()
        self.pos_w = pos_w; self.smooth = smooth
    def forward(self, logits, targets):
        pw  = torch.tensor(self.pos_w, device=logits.device)
        bce = nn.functional.binary_cross_entropy_with_logits(
                  logits, targets, pos_weight=pw, reduction="mean")
        p   = torch.sigmoid(logits)
        inter = (p * targets).sum()
        dice  = 1.0 - (2*inter + self.smooth) / (p.sum() + targets.sum() + self.smooth)
        return bce + dice

print("Model & Loss defined.")


In [ ]:

train_df = meta[meta.split == "train"].sample(frac=1, random_state=42).reset_index(drop=True)
val_df   = train_df.iloc[:200].reset_index(drop=True)
train_df = train_df.iloc[200:].reset_index(drop=True)
print(f"Train: {len(train_df)}   Val: {len(val_df)}")

train_ds = ApoDataset(train_df, aug=True)
val_ds   = ApoDataset(val_df,   aug=False)
train_dl = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

model     = UNetR34().to(DEVICE)
criterion = BCEDiceLoss(pos_w=2.0)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)
scaler    = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

EPOCHS = 40
CKPT   = "/kaggle/working/best_apo_model.pth"
best_dice = -1.0

def dice_coef(prob, tgt, thresh=0.5):
    p = (prob > thresh).float()
    inter = (p * tgt).sum()
    return ((2*inter + 1e-5) / (p.sum() + tgt.sum() + 1e-5)).item()

for epoch in range(1, EPOCHS + 1):
    model.train()
    run_loss = 0.0
    for imgs, msks in train_dl:
        imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            loss = criterion(model(imgs), msks)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        run_loss += loss.item() * len(imgs)
    scheduler.step()

    if epoch % 5 == 0 or epoch == 1:
        model.eval()
        dv = []
        with torch.no_grad():
            for imgs, msks in val_dl:
                imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
                dv.append(dice_coef(torch.sigmoid(model(imgs)), msks))
        d = float(np.mean(dv))
        print(f"Ep {epoch:03d} | loss {run_loss/len(train_ds):.4f} | "
              f"apo Dice {d:.4f} | lr {scheduler.get_last_lr()[0]:.2e}")
        if d > best_dice:
            best_dice = d; torch.save(model.state_dict(), CKPT)
            print(f"  ** saved (dice={d:.4f})")

print(f"\nBest val Dice = {best_dice:.4f}")
model.load_state_dict(torch.load(CKPT, map_location=DEVICE)); model.eval()
print("Best checkpoint loaded.")


In [ ]:

# Geometry: all in ORIGINAL image pixels.
# PA = atan(MT / dx)   FL = MT / sin(PA)

def fit_line_huber(pts_yx):
    """cv2 Huber line fit. pts_yx: (N,2) float32 [y,x]."""
    if len(pts_yx) < 15: return None
    pts = pts_yx[:, ::-1].astype(np.float32)   # to [x,y]
    line = cv2.fitLine(pts.reshape(-1, 1, 2), cv2.DIST_HUBER, 0, 0.01, 0.01)
    vx, vy, x0, y0 = line.flatten()
    return float(vx), float(vy), float(x0), float(y0)

def to_mb(vx, vy, x0, y0):
    """Convert fitLine output to (m, b) where y = m*x + b."""
    if abs(vx) < 1e-9: return None
    m = vy / vx; b = y0 - m * x0
    return float(m), float(b)

def perp_dist_mean(m_dep, b_dep, m_sup, b_sup, x_lo, x_hi):
    xs = np.linspace(x_lo, x_hi, 200)
    y_sup = m_sup * xs + b_sup
    d = np.abs(m_dep * xs - y_sup + b_dep) / math.sqrt(m_dep**2 + 1)
    return float(d.mean())

def structure_tensor_angle(img_hw, band_mask, sigma_pre=2, sigma_post=12):
    """Coherence-weighted structure tensor. Returns fascicle angle (deg) or None."""
    bl = cv2.GaussianBlur(img_hw, (0, 0), sigma_pre)
    gx = cv2.Sobel(bl, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(bl, cv2.CV_32F, 0, 1, ksize=3)
    Jxx = cv2.GaussianBlur(gx*gx, (0, 0), sigma_post)
    Jyy = cv2.GaussianBlur(gy*gy, (0, 0), sigma_post)
    Jxy = cv2.GaussianBlur(gx*gy, (0, 0), sigma_post)
    diff = Jxx - Jyy
    coh  = np.sqrt(diff**2 + 4*Jxy**2) / (Jxx + Jyy + 1e-8)
    w = np.where(band_mask, coh, 0.0)
    tw = w.sum()
    if tw < 1e-6: return None
    Jxx_w = (w * Jxx).sum() / tw
    Jyy_w = (w * Jyy).sum() / tw
    Jxy_w = (w * Jxy).sum() / tw
    # gradient direction (perpendicular to fascicles)
    theta_g = 0.5 * math.atan2(2*Jxy_w, Jxx_w - Jyy_w)
    theta_f = math.degrees(theta_g) + 90.0
    while theta_f >=  90: theta_f -= 180
    while theta_f < -90:  theta_f += 180
    return theta_f

def wrap_pa(theta_fasc, theta_deep):
    d = theta_fasc - theta_deep
    while d >  90: d -= 180
    while d < -90: d += 180
    return abs(d)

def measure_image(apo_prob_s, orig_img, orig_w, orig_h, margins=(0.10, 0.05, 0.15)):
    """Returns (pa_deg, mt_px) or None. All in original image pixels."""
    apo_f   = cv2.resize(apo_prob_s, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
    apo_bin = (apo_f > 0.40).astype(np.uint8)

    lbl, n = ndimage.label(apo_bin)
    if n < 2: return None
    sizes = [(lbl == i).sum() for i in range(1, n+1)]
    top2  = sorted(range(1, n+1), key=lambda i: sizes[i-1], reverse=True)[:2]
    top2  = sorted(top2, key=lambda i: np.where(lbl==i)[0].mean())
    sup_yx = np.column_stack(np.where(lbl == top2[0])).astype(np.float32)
    dep_yx = np.column_stack(np.where(lbl == top2[1])).astype(np.float32)

    sl = fit_line_huber(sup_yx); dl = fit_line_huber(dep_yx)
    if sl is None or dl is None: return None
    smb = to_mb(*sl); dmb = to_mb(*dl)
    if smb is None or dmb is None: return None
    m_sup, b_sup = smb; m_dep, b_dep = dmb

    x_lo = max(sup_yx[:,1].min(), dep_yx[:,1].min())
    x_hi = min(sup_yx[:,1].max(), dep_yx[:,1].max())
    if x_hi - x_lo < orig_w * 0.10: return None

    mt_px = perp_dist_mean(m_dep, b_dep, m_sup, b_sup, x_lo, x_hi)
    theta_deep = math.degrees(math.atan(m_dep))

    ys = np.arange(orig_h, dtype=np.float32)
    xs = np.arange(orig_w, dtype=np.float32)
    XS, YS = np.meshgrid(xs, ys)
    y_sup_grid = m_sup * XS + b_sup
    y_dep_grid = m_dep * XS + b_dep

    for mf in margins:
        mg = mt_px * mf
        y_lo = np.minimum(y_sup_grid, y_dep_grid) + mg
        y_hi = np.maximum(y_sup_grid, y_dep_grid) - mg
        band = (YS >= y_lo) & (YS <= y_hi)
        if band.sum() < 500: continue
        tf = structure_tensor_angle(orig_img, band, sigma_pre=2, sigma_post=12)
        if tf is None: continue
        pa = wrap_pa(tf, theta_deep)
        if pa < 3.0: continue
        return pa, mt_px

    return None

print("Geometry engine ready.")


In [ ]:

print("Computing fallback medians from GT aponeurosis masks ...")

fallback_pa, fallback_mt = [], []
sample_rows = meta[meta.split == "train"].sample(min(400, (meta.split=="train").sum()),
                                                  random_state=0)

for _, row in sample_rows.iterrows():
    fname  = row.image_id; orig_w, orig_h = int(row.orig_w), int(row.orig_h)
    stem   = os.path.splitext(fname)[0]
    mmpx   = get_mmpx(stem, orig_w, orig_h)

    gt_s   = load_mask_np(os.path.join(APO_MSK_DIR, fname))
    gt_img = np.array(Image.open(os.path.join(APO_IMG_DIR, fname)).convert("L"),
                       dtype=np.float32) / 255.0

    res = measure_image(gt_s, gt_img, orig_w, orig_h)
    if res is None: continue
    pa, mt_px = res
    fallback_pa.append(pa); fallback_mt.append(mt_px * mmpx)

FALLBACK_PA = float(np.median(fallback_pa)) if fallback_pa else 14.871
FALLBACK_MT = float(np.median(fallback_mt)) if fallback_mt else 19.368
FALLBACK_FL = float(np.median(fallback_fl)) if 'fallback_fl' in locals() and fallback_fl else 99.514
print(f"Fallback PA={FALLBACK_PA:.2f} deg  MT={FALLBACK_MT:.2f} mm  FL={FALLBACK_FL:.2f} mm")
print(f"(derived from {len(fallback_pa)} GT images)")


In [ ]:

print("Proxy MAE: GT masks vs model predictions on val split ...")
model.eval()
gt_pas, gt_mts, pr_pas, pr_mts = [], [], [], []

for _, row in val_df.iterrows():
    fname  = row.image_id; orig_w, orig_h = int(row.orig_w), int(row.orig_h)
    stem   = os.path.splitext(fname)[0]; mmpx = get_mmpx(stem, orig_w, orig_h)
    orig_img = np.array(Image.open(os.path.join(APO_IMG_DIR, fname)).convert("L"),
                         dtype=np.float32) / 255.0
    gt_s    = load_mask_np(os.path.join(APO_MSK_DIR, fname))
    res_gt  = measure_image(gt_s, orig_img, orig_w, orig_h)

    img_t  = load_img_tensor(os.path.join(APO_IMG_DIR, fname)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = torch.sigmoid(model(img_t))[0, 0].cpu().numpy()
    res_pr = measure_image(prob, orig_img, orig_w, orig_h)

    if res_gt is None or res_pr is None: continue
    gt_pas.append(res_gt[0]); gt_mts.append(res_gt[1] * mmpx)
    pr_pas.append(res_pr[0]); pr_mts.append(res_pr[1] * mmpx)

if gt_pas:
    def to_fl(pa, mt):
        return mt / math.sin(math.radians(pa)) if pa > 3 else None
    pairs = [(to_fl(gp, gm), to_fl(pp, pm))
             for gp, gm, pp, pm in zip(gt_pas, gt_mts, pr_pas, pr_mts)
             if to_fl(gp, gm) and to_fl(pp, pm)]
    print(f"Proxy MAE on {len(gt_pas)} val images:")
    print(f"  PA: {np.mean(np.abs(np.array(gt_pas)-np.array(pr_pas))):.3f} deg")
    print(f"  MT: {np.mean(np.abs(np.array(gt_mts)-np.array(pr_mts))):.3f} mm")
    print(f"  FL: {np.mean(np.abs(np.array([g for g,_ in pairs])-np.array([p for _,p in pairs]))):.3f} mm")
else:
    print("Not enough paired results.")


In [ ]:

test_sub = meta[meta.split == "test"].reset_index(drop=True)
test_ds  = ApoDataset(test_sub, aug=False)
test_dl  = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=2)

model.eval()
rows = []; n_fallback = 0

print(f"Inference on {len(test_sub)} test images ...")

for imgs, fnames, orig_ws, orig_hs in test_dl:
    imgs = imgs.to(DEVICE)
    with torch.no_grad():
        probs = torch.sigmoid(model(imgs)).cpu().numpy()   # (B,1,H,W)

    for i, fname in enumerate(fnames):
        orig_w = int(orig_ws[i]); orig_h = int(orig_hs[i])
        stem   = os.path.splitext(fname)[0]
        mmpx   = get_mmpx(stem, orig_w, orig_h)
        apo_p  = probs[i, 0]

        orig_img = np.array(Image.open(os.path.join(TEST_IMG_DIR, fname)).convert("L"),
                             dtype=np.float32) / 255.0

        result = measure_image(apo_p, orig_img, orig_w, orig_h)

        if result is not None:
            pa, mt_px = result
            mt_mm = mt_px * mmpx
            fl_mm = mt_mm / math.sin(math.radians(pa))
        else:
            pa, fl_mm, mt_mm = FALLBACK_PA, FALLBACK_FL, FALLBACK_MT
            n_fallback += 1

        rows.append({"image_id": fname, "pa_deg": round(pa, 3),
                     "fl_mm": round(fl_mm, 3), "mt_mm": round(mt_mm, 3)})

sub = pd.DataFrame(rows)
sub.to_csv("/kaggle/working/submission.csv", index=False, sep=",")

print(f"\n[SUCCESS] submission.csv: {len(sub)} rows")
print(f"Fallback: {n_fallback}/{len(sub)} ({100*n_fallback/len(sub):.1f}%)")
print(sub.describe())
print("\n-- First 6 raw lines --")
with open("/kaggle/working/submission.csv") as f:
    for _ in range(6): print(f.readline().rstrip())
